# Coil Analysis Notebook
### Goal: identify coil cases, determine if coil was opened, extract resolution, compare labour hours.

**Step 1 — Validate keyword detection** against the 106 NAM-labeled coil cases before scaling to the full dataset.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/processed/cfr_savings_processed.parquet")
print(f"Total cases: {len(df)}")
print(f"Countries: {df['country'].nunique()}")
print(f"Markets: {df['market'].value_counts().to_dict()}")

Total cases: 13194
Countries: 53
Markets: {'NAM': 2310, 'LAT': 1996, 'ISC': 1537, 'DAC': 1346, 'META': 1039, 'IIG': 983, 'IBE': 839, 'UKI': 698, 'JPN': 555, 'APA': 526, 'GRC': 407, 'FRA': 292, 'BNL': 204, 'NOR': 181, 'RCA': 129, 'CEE': 85, 'Other': 67}


## Translation coverage check
Before running keyword search, verify how many cases have translated text available.
Cases without translation will need to be handled separately.

In [2]:
# check fill rates for the text columns we'll search
text_cols = ["remote_remarks_en", "field_remarks_en"]

print("TRANSLATION COVERAGE")
print("-" * 60)
for col in text_cols:
    filled = df[col].notna().sum()
    empty = df[col].isna().sum()
    print(f"  {col:<25} {filled:>6}/{len(df)} ({filled/len(df):.1%}) filled, {empty} missing")

# check if missing translations correlate with specific markets
print("\nMISSING TRANSLATIONS BY MARKET")
print("-" * 60)
for col in text_cols:
    missing = df[df[col].isna()]
    if len(missing) > 0:
        print(f"\n  {col}:")
        print(missing["market"].value_counts().to_string())

# cases with at least one translated remark
has_any_text = df["remote_remarks_en"].notna() | df["field_remarks_en"].notna()
print(f"\nCases with at least one translated remark: {has_any_text.sum()}/{len(df)} ({has_any_text.mean():.1%})")
print(f"Cases with NO translated text at all: {(~has_any_text).sum()}")

TRANSLATION COVERAGE
------------------------------------------------------------
  remote_remarks_en          12318/13194 (93.4%) filled, 876 missing
  field_remarks_en            7630/13194 (57.8%) filled, 5564 missing

MISSING TRANSLATIONS BY MARKET
------------------------------------------------------------

  remote_remarks_en:
market
NAM     289
APA     134
LAT     132
DAC      73
UKI      64
IBE      59
IIG      44
META     21
GRC      19
ISC      10
FRA       9
CEE       9
NOR       8
BNL       3
RCA       2

  field_remarks_en:
market
LAT      1384
NAM       848
META      602
DAC       537
IBE       417
ISC       416
IIG       393
UKI       303
FRA       194
APA       163
NOR        89
BNL        89
Other      49
GRC        38
CEE        31
RCA         7
JPN         4

Cases with at least one translated remark: 13091/13194 (99.2%)
Cases with NO translated text at all: 103


## NAM-labeled coil cases (ground truth)
These 106 cases are confirmed coil cases by NAM engineers.
We'll use them to validate our keyword detection.

In [3]:
# the labeled coil cases
labeled_coil = df[df["nam_main_category"] == "coil"]
print(f"NAM-labeled coil cases: {len(labeled_coil)}")

# sub-category distribution within coil
if "nam_sub_category" in df.columns:
    print(f"\nCoil sub-categories:")
    print(labeled_coil["nam_sub_category"].value_counts().to_string())

# quick look at what text is available for these cases
print(f"\nText availability in labeled coil cases:")
for col in ["remote_remarks_en", "field_remarks_en"]:
    filled = labeled_coil[col].notna().sum()
    print(f"  {col}: {filled}/{len(labeled_coil)}")

NAM-labeled coil cases: 106

Coil sub-categories:
nam_sub_category
dstream               79
digital               15
other                  6
analog_non_synergy     3
fse_aware              1

Text availability in labeled coil cases:
  remote_remarks_en: 104/106
  field_remarks_en: 77/106


## Keyword detection
Search for "coil" in raw translated remarks (remote + field).
Raw remarks are the most comprehensive — they contain everything the extracted fields do, plus more.

In [4]:
def contains_keyword(text, keyword="coil"):
    """Case-insensitive check if keyword appears in text as a whole word."""
    if pd.isna(text):
        return False
    # \b word boundary avoids matching e.g. "recoil" or "coiled"
    import re
    return bool(re.search(r"\b" + keyword + r"\b", str(text), re.IGNORECASE))


# search across both remote and field remarks
df["coil_in_remote"] = df["remote_remarks_en"].apply(contains_keyword)
df["coil_in_field"] = df["field_remarks_en"].apply(contains_keyword)
df["coil_detected"] = df["coil_in_remote"] | df["coil_in_field"]

print("KEYWORD DETECTION RESULTS")
print("-" * 60)
print(f"  'coil' found in remote remarks: {df['coil_in_remote'].sum()}")
print(f"  'coil' found in field remarks:  {df['coil_in_field'].sum()}")
print(f"  'coil' found in either:         {df['coil_detected'].sum()}")
print(f"  Total cases:                    {len(df)}")

KEYWORD DETECTION RESULTS
------------------------------------------------------------
  'coil' found in remote remarks: 1736
  'coil' found in field remarks:  1096
  'coil' found in either:         2131
  Total cases:                    13194


## Validation: keyword vs labeled coil cases
Check recall (how many labeled coil cases does keyword catch?) and
false positive rate (how many keyword hits are NOT coil cases?).

In [5]:
# ── Recall: how many of the 106 labeled coil cases did we catch? ──
labeled_coil_detected = df[df["nam_main_category"] == "coil"]["coil_detected"].sum()
labeled_coil_total = len(df[df["nam_main_category"] == "coil"])
recall = labeled_coil_detected / labeled_coil_total

print("RECALL (on NAM-labeled coil cases)")
print("-" * 60)
print(f"  Detected: {labeled_coil_detected}/{labeled_coil_total} ({recall:.1%})")

# ── False negatives: labeled coil cases NOT caught by keyword ──
false_negatives = df[(df["nam_main_category"] == "coil") & (~df["coil_detected"])]
print(f"  Missed (false negatives): {len(false_negatives)}")

# ── Among NAM-labeled cases: precision ──
# How many keyword-detected cases (within labeled data) are actually coil?
nam_labeled = df[df["nam_main_category"].notna()]
detected_in_labeled = nam_labeled[nam_labeled["coil_detected"]]
true_positives = detected_in_labeled[detected_in_labeled["nam_main_category"] == "coil"]
false_positives = detected_in_labeled[detected_in_labeled["nam_main_category"] != "coil"]

precision = len(true_positives) / len(detected_in_labeled) if len(detected_in_labeled) > 0 else 0
print(f"\nPRECISION (within NAM-labeled cases only)")
print("-" * 60)
print(f"  Keyword detected: {len(detected_in_labeled)} labeled cases")
print(f"  True positives (actually coil):  {len(true_positives)}")
print(f"  False positives (not coil):      {len(false_positives)}")
print(f"  Precision: {precision:.1%}")

RECALL (on NAM-labeled coil cases)
------------------------------------------------------------
  Detected: 105/106 (99.1%)
  Missed (false negatives): 1

PRECISION (within NAM-labeled cases only)
------------------------------------------------------------
  Keyword detected: 469 labeled cases
  True positives (actually coil):  105
  False positives (not coil):      364
  Precision: 22.4%


## Inspect false negatives
Labeled coil cases missed by keyword search — why?

In [6]:
if len(false_negatives) > 0:
    print(f"Inspecting {len(false_negatives)} false negatives:\n")
    for i, (_, row) in enumerate(false_negatives.iterrows()):
        print(f"  Case {i+1} | sub_category: {row.get('nam_sub_category', 'N/A')}")
        remote = str(row.get("remote_remarks_en", ""))[:300] if pd.notna(row.get("remote_remarks_en")) else "EMPTY"
        field = str(row.get("field_remarks_en", ""))[:300] if pd.notna(row.get("field_remarks_en")) else "EMPTY"
        print(f"  Remote: {remote}")
        print(f"  Field:  {field}")
        print()
else:
    print("No false negatives — keyword search caught all labeled coil cases.")

Inspecting 1 false negatives:

  Case 1 | sub_category: nan
  Remote: *** Diagnostic performed by Engineer [2025-06-11 20:25:57]
Looked at logs downloaded via telnet - Showed low flow in gradient amp circuit, T1 
Chiller temp and flow were good,
GA circuit temp had risen from 20,4C to 25C indicating possible low flow not allowing heat exchanger to work properly,
Techn
  Field:  EMPTY



## Inspect false positives
Cases where "coil" appears in text but the case is labeled as something else.
This helps determine if the keyword is too broad.

In [7]:
if len(false_positives) > 0:
    print(f"Inspecting false positives ({len(false_positives)} cases):")
    print(f"\nTrue categories of false positives:")
    print(false_positives["nam_main_category"].value_counts().to_string())

    print(f"\n{'=' * 60}")
    print("SAMPLE FALSE POSITIVES")
    print("=" * 60)
    for i, (_, row) in enumerate(false_positives.head(10).iterrows()):
        print(f"\n  Case {i+1} | True label: {row['nam_main_category']} / {row.get('nam_sub_category', 'N/A')}")
        # show the sentence containing "coil" for context
        import re
        for col in ["remote_remarks_en", "field_remarks_en"]:
            text = str(row.get(col, ""))
            if pd.notna(row.get(col)):
                # find sentences containing "coil"
                sentences = re.split(r'[.\n]', text)
                coil_sentences = [s.strip() for s in sentences if re.search(r'\bcoil\b', s, re.IGNORECASE)]
                if coil_sentences:
                    source = "Remote" if col == "remote_remarks_en" else "Field"
                    for s in coil_sentences[:2]:
                        print(f"  {source}: ...{s[:200]}...")
else:
    print("No false positives within labeled data.")

Inspecting false positives (364 cases):

True categories of false positives:
nam_main_category
cooling_system      66
patient_support     59
data_acquisition    56
rf_amp              51
gradient_amp        40
magnet              22
software            16
chiller             12
image_quality       11
consoles            11
reconstructor        8
environmental        7
other                4
third_party_item     1

SAMPLE FALSE POSITIVES

  Case 1 | True label: environmental / power
  Remote: ...Customer ran a head coil scan and system working,...

  Case 2 | True label: software / applications
  Remote: ...How was the device being used? In clinical use, Customer reported that the system keeps aborting mid scan on multiple back to back L-spine exams with the spine coil being used, Real time error occurre...

  Case 3 | True label: rf_amp / rf_amp
  Remote: ...Run Test scan on 3L bottle with body coil...

  Case 4 | True label: patient_support / table_lockup
  Remote: ...Checked logs and

## Detection summary

In [8]:
print("=" * 60)
print("  KEYWORD DETECTION SUMMARY")
print("=" * 60)
print(f"\n  Recall on labeled data:    {recall:.1%} ({labeled_coil_detected}/{labeled_coil_total})")
print(f"  Precision on labeled data: {precision:.1%} ({len(true_positives)}/{len(detected_in_labeled)})")
print(f"  False negatives:           {len(false_negatives)}")
print(f"  False positives:           {len(false_positives)}")
print(f"\n  Total coil-detected cases (full dataset): {df['coil_detected'].sum()}")
print(f"\n  Detection by source:")
print(f"    Remote only: {(df['coil_in_remote'] & ~df['coil_in_field']).sum()}")
print(f"    Field only:  {(~df['coil_in_remote'] & df['coil_in_field']).sum()}")
print(f"    Both:        {(df['coil_in_remote'] & df['coil_in_field']).sum()}")

print(f"\n  Detected cases by market:")
print(df[df["coil_detected"]]["market"].value_counts().to_string())

  KEYWORD DETECTION SUMMARY

  Recall on labeled data:    99.1% (105/106)
  Precision on labeled data: 22.4% (105/469)
  False negatives:           1
  False positives:           364

  Total coil-detected cases (full dataset): 2131

  Detection by source:
    Remote only: 1035
    Field only:  395
    Both:        701

  Detected cases by market:
market
NAM      550
ISC      307
DAC      274
LAT      261
UKI      123
IIG      113
IBE       95
META      87
APA       74
JPN       68
GRC       49
NOR       42
BNL       32
FRA       19
RCA       18
CEE       14
Other      5


---
## Next steps
Based on the validation results above:
- If recall and precision are high → proceed with keyword detection for the full dataset
- If false negatives are significant → consider adding synonym terms or LLM-based detection
- If false positives are significant → add filtering rules or LLM confirmation

**Step 2** (next cells): For detected coil cases, call LLM to determine:
1. Was the coil physically opened?
2. What was the resolution?

# Improve Precisoion with Filter where coil appears

In [9]:
# ── Cell 1: Define problem-focused vs context-only fields ──

import re

def has_coil(text):
    """Check if 'coil' appears as a whole word."""
    if pd.isna(text):
        return False
    return bool(re.search(r"\bcoil\b", str(text), re.IGNORECASE))

# Problem-focused fields: if coil appears here, the case is likely ABOUT a coil
problem_fields = [
    "extracted_problem_description_field",
    "extracted_problem_description_remote",
    "extracted_malfunction_area_field",
    "extracted_malfunction_area_remote",
    "extracted_error_field",
    "extracted_error_remote",
]

# Context fields: coil might just be mentioned in passing
context_fields = [
    "extracted_diagnostic_field",
    "extracted_diagnostic_remote",
    "extracted_repair_action_field",
    "extracted_repair_action_remote",
    "extracted_troubleshooting_field",
    "extracted_troubleshooting_remote",
]

# check which fields actually exist in the dataframe
problem_fields = [f for f in problem_fields if f in df.columns]
context_fields = [f for f in context_fields if f in df.columns]

print(f"Problem-focused fields found: {problem_fields}")
print(f"Context fields found: {context_fields}")

Problem-focused fields found: ['extracted_problem_description_field', 'extracted_problem_description_remote', 'extracted_malfunction_area_field', 'extracted_malfunction_area_remote', 'extracted_error_field', 'extracted_error_remote']
Context fields found: ['extracted_diagnostic_field', 'extracted_diagnostic_remote', 'extracted_repair_action_field', 'extracted_repair_action_remote', 'extracted_troubleshooting_field', 'extracted_troubleshooting_remote']


In [10]:
# ── Cell 2: Classify where coil appears ──

# only look at keyword-detected cases
coil_candidates = df[df["coil_detected"]].copy()
print(f"Keyword-detected cases to analyze: {len(coil_candidates)}")

# check if coil appears in any problem-focused field
coil_candidates["coil_in_problem"] = coil_candidates[problem_fields].apply(
    lambda row: any(has_coil(row[col]) for col in problem_fields), axis=1
)

# check if coil appears in any context-only field
coil_candidates["coil_in_context"] = coil_candidates[context_fields].apply(
    lambda row: any(has_coil(row[col]) for col in context_fields), axis=1
)

# classification:
# - "problem": coil in problem-focused field (strong signal)
# - "context_only": coil only in context fields (weak signal)
# - "raw_only": coil only in raw remarks, not in any extracted field
coil_candidates["coil_location"] = "raw_only"
coil_candidates.loc[coil_candidates["coil_in_context"], "coil_location"] = "context_only"
coil_candidates.loc[coil_candidates["coil_in_problem"], "coil_location"] = "problem"

print(f"\nCoil location breakdown:")
print(coil_candidates["coil_location"].value_counts().to_string())

Keyword-detected cases to analyze: 2131

Coil location breakdown:
coil_location
context_only    1021
problem          671
raw_only         439


In [11]:
# ── Cell 3: Validate against labeled data ──

# filter to labeled cases only
labeled_candidates = coil_candidates[coil_candidates["nam_main_category"].notna()].copy()
labeled_candidates["is_coil"] = labeled_candidates["nam_main_category"] == "coil"

print("PRECISION BY COIL LOCATION (within labeled data)")
print("=" * 60)

for location in ["problem", "context_only", "raw_only"]:
    subset = labeled_candidates[labeled_candidates["coil_location"] == location]
    if len(subset) == 0:
        print(f"  {location:<15} — no cases")
        continue
    tp = subset["is_coil"].sum()
    total = len(subset)
    precision = tp / total
    print(f"  {location:<15} {tp:>4} coil / {total:>4} total → precision {precision:.1%}")

# if we only keep "problem" cases, what's our recall?
problem_only = labeled_candidates[labeled_candidates["coil_location"] == "problem"]
true_coil_in_problem = problem_only["is_coil"].sum()
total_labeled_coil = labeled_candidates["is_coil"].sum()
recall = true_coil_in_problem / total_labeled_coil if total_labeled_coil > 0 else 0

print(f"\nIf we keep ONLY 'problem' cases:")
print(f"  Recall:    {recall:.1%} ({true_coil_in_problem}/{total_labeled_coil})")
print(f"  Precision: {true_coil_in_problem}/{len(problem_only)} ({true_coil_in_problem/len(problem_only):.1%})" if len(problem_only) > 0 else "  No problem cases")

# what about problem + context_only?
problem_or_context = labeled_candidates[labeled_candidates["coil_location"].isin(["problem", "context_only"])]
tp2 = problem_or_context["is_coil"].sum()
recall2 = tp2 / total_labeled_coil if total_labeled_coil > 0 else 0

print(f"\nIf we keep 'problem' + 'context_only':")
print(f"  Recall:    {recall2:.1%} ({tp2}/{total_labeled_coil})")
print(f"  Precision: {tp2}/{len(problem_or_context)} ({tp2/len(problem_or_context):.1%})" if len(problem_or_context) > 0 else "  No cases")

PRECISION BY COIL LOCATION (within labeled data)
  problem           56 coil /  135 total → precision 41.5%
  context_only      45 coil /  238 total → precision 18.9%
  raw_only           4 coil /   96 total → precision 4.2%

If we keep ONLY 'problem' cases:
  Recall:    53.3% (56/105)
  Precision: 56/135 (41.5%)

If we keep 'problem' + 'context_only':
  Recall:    96.2% (101/105)
  Precision: 101/373 (27.1%)


In [12]:
# ── Cell 4: Impact on full dataset ──

print("FULL DATASET — EXPECTED COIL CASES BY FILTER")
print("=" * 60)

for location in ["problem", "context_only", "raw_only"]:
    n = (coil_candidates["coil_location"] == location).sum()
    print(f"  {location:<15} {n:>5} cases")

problem_total = (coil_candidates["coil_location"] == "problem").sum()
combined_total = coil_candidates["coil_location"].isin(["problem", "context_only"]).sum()

print(f"\n  'problem' only:              {problem_total} cases to analyze")
print(f"  'problem' + 'context_only':  {combined_total} cases to analyze")
print(f"  all keyword-detected:        {len(coil_candidates)} cases to analyze")

FULL DATASET — EXPECTED COIL CASES BY FILTER
  problem           671 cases
  context_only     1021 cases
  raw_only          439 cases

  'problem' only:              671 cases to analyze
  'problem' + 'context_only':  1692 cases to analyze
  all keyword-detected:        2131 cases to analyze


# Improve precision with LLM 

In [13]:
# ── Cell 1: LLM prompt and parser ──

import json
import time
from src.classification.classifier import call_llm

COIL_ANALYSIS_PROMPT = """You are an expert MRI service engineer analyzing a technical service call.

Answer these three questions about the case below.

QUESTION 1 — Is this case primarily about a coil problem?
A coil problem means the coil (RF body coil, head coil, surface coil, etc.) is the root cause
or main subject of the service call. Cases that merely mention a coil while addressing a
different problem (e.g. gradient, magnet, software) should be answered "no".

QUESTION 2 — Was the coil physically opened for inspection or repair?
"Opened" means the coil housing was physically opened to access internal components.
If the case is not about a coil, answer "not_applicable".

QUESTION 3 — What was the resolution?
Briefly describe what action resolved the issue (e.g. "replaced coil element",
"recalibrated via software", "reseated connector"). If unclear, answer "unclear".
If the case is not about a coil, answer "not_applicable".

CASE TEXT:
"{case_text}"

RESPOND IN EXACTLY THIS JSON FORMAT (nothing else):
{{"is_coil_case": true/false, "coil_opened": true/false/null, "resolution": "brief description"}}"""


def parse_coil_response(response_text):
    """Parse the LLM JSON response. Returns dict with defaults on failure."""
    default = {"is_coil_case": None, "coil_opened": None, "resolution": None, "parse_error": True}
    try:
        # strip markdown fences if present
        clean = response_text.strip().replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)
        parsed["parse_error"] = False
        return parsed
    except (json.JSONDecodeError, KeyError):
        return default

In [14]:
# ── Cell 2: Build case texts for labeled candidates ──

# use the same LLM context strategy: prefer raw remarks, fallback to extracted fields
def build_coil_case_text(row):
    """Build text for the coil analysis prompt. Uses both remote + field for completeness."""
    parts = []

    # remote remarks
    remote = row.get("remote_remarks_en")
    if pd.notna(remote) and len(str(remote).strip()) > 20:
        parts.append(f"REMOTE REMARKS:\n{str(remote).strip()}")

    # field remarks
    field = row.get("field_remarks_en")
    if pd.notna(field) and len(str(field).strip()) > 20:
        parts.append(f"FIELD REMARKS:\n{str(field).strip()}")

    return "\n\n".join(parts) if parts else None


labeled_candidates = coil_candidates[coil_candidates["nam_main_category"].notna()].copy()
labeled_candidates["case_text"] = labeled_candidates.apply(build_coil_case_text, axis=1)

# drop cases with no text
has_text = labeled_candidates["case_text"].notna()
print(f"Labeled candidates with text: {has_text.sum()}/{len(labeled_candidates)}")
labeled_candidates = labeled_candidates[has_text].copy()

Labeled candidates with text: 469/469


In [15]:
# ── Cell 3: Run LLM on labeled candidates ──

print(f"Running LLM analysis on {len(labeled_candidates)} labeled candidates...")
print(f"Estimated time: ~{len(labeled_candidates) * 2 / 60:.0f} minutes\n")

results_list = []

for i, (idx, row) in enumerate(labeled_candidates.iterrows()):
    prompt = COIL_ANALYSIS_PROMPT.format(case_text=row["case_text"][:3000])
    
    try:
        response = call_llm(prompt)
        parsed = parse_coil_response(response)
    except Exception as e:
        parsed = {"is_coil_case": None, "coil_opened": None, "resolution": None, "parse_error": True}
        print(f"  ERROR at case {i+1}: {e}")

    parsed["df_index"] = idx
    results_list.append(parsed)

    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(labeled_candidates)}...")

print(f"  Done: {len(results_list)} cases processed")

# merge results back
llm_results = pd.DataFrame(results_list).set_index("df_index")
labeled_candidates["llm_is_coil"] = llm_results["is_coil_case"]
labeled_candidates["llm_coil_opened"] = llm_results["coil_opened"]
labeled_candidates["llm_resolution"] = llm_results["resolution"]
labeled_candidates["llm_parse_error"] = llm_results["parse_error"]

parse_errors = labeled_candidates["llm_parse_error"].sum()
print(f"\nParse errors: {parse_errors}/{len(labeled_candidates)}")

Running LLM analysis on 469 labeled candidates...
Estimated time: ~16 minutes

  Processed 50/469...
  Processed 100/469...
  Processed 150/469...
  Processed 200/469...
  Processed 250/469...
  Processed 300/469...
  Processed 350/469...
  Processed 400/469...
  Processed 450/469...
  Done: 469 cases processed

Parse errors: 0/469


In [16]:
# ── Cell 4: Validate coil detection ──

# ground truth: NAM label says "coil"
labeled_candidates["true_is_coil"] = labeled_candidates["nam_main_category"] == "coil"

# filter out parse errors for clean evaluation
valid = labeled_candidates[~labeled_candidates["llm_parse_error"]].copy()
print(f"Valid responses: {len(valid)}/{len(labeled_candidates)}")

tp = ((valid["llm_is_coil"] == True) & (valid["true_is_coil"] == True)).sum()
fp = ((valid["llm_is_coil"] == True) & (valid["true_is_coil"] == False)).sum()
fn = ((valid["llm_is_coil"] == False) & (valid["true_is_coil"] == True)).sum()
tn = ((valid["llm_is_coil"] == False) & (valid["true_is_coil"] == False)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nLLM COIL DETECTION (on keyword-filtered labeled cases)")
print("=" * 60)
print(f"  True positives:  {tp}")
print(f"  False positives: {fp}")
print(f"  False negatives: {fn}")
print(f"  True negatives:  {tn}")
print(f"\n  Precision: {precision:.1%}")
print(f"  Recall:    {recall:.1%}")
print(f"  F1:        {f1:.1%}")

# comparison with keyword-only approach
print(f"\n  COMPARISON")
print(f"  {'Method':<20} {'Precision':>10} {'Recall':>10}")
print(f"  {'-'*42}")
print(f"  {'Keyword only':<20} {'22.4%':>10} {'99.1%':>10}")
print(f"  {'Keyword + LLM':<20} {precision:>9.1%} {recall:>9.1%}")

Valid responses: 469/469

LLM COIL DETECTION (on keyword-filtered labeled cases)
  True positives:  99
  False positives: 77
  False negatives: 6
  True negatives:  287

  Precision: 56.2%
  Recall:    94.3%
  F1:        70.5%

  COMPARISON
  Method                Precision     Recall
  ------------------------------------------
  Keyword only              22.4%      99.1%
  Keyword + LLM            56.2%     94.3%


In [17]:
# ── Cell 5: Inspect LLM disagreements ──

print("LLM FALSE POSITIVES — LLM says coil, label says otherwise")
print("=" * 60)
fp_cases = valid[(valid["llm_is_coil"] == True) & (valid["true_is_coil"] == False)]
for i, (_, row) in enumerate(fp_cases.head(5).iterrows()):
    print(f"\n  Case {i+1} | True label: {row['nam_main_category']}")
    print(f"  LLM resolution: {row['llm_resolution']}")
    text = str(row["case_text"])[:200]
    print(f"  Text: {text}...")

print(f"\n\nLLM FALSE NEGATIVES — LLM says not coil, label says coil")
print("=" * 60)
fn_cases = valid[(valid["llm_is_coil"] == False) & (valid["true_is_coil"] == True)]
for i, (_, row) in enumerate(fn_cases.head(5).iterrows()):
    print(f"\n  Case {i+1} | Sub-category: {row.get('nam_sub_category', 'N/A')}")
    print(f"  Text: {str(row['case_text'])[:200]}...")

LLM FALSE POSITIVES — LLM says coil, label says otherwise

  Case 1 | True label: patient_support
  LLM resolution: Reset table / instructed to move table to the outmost horizontal position and press Resume to re-engage the posterior coil; also advised checking emergency stop chain and patient support diagnostics
  Text: REMOTE REMARKS:
*** Diagnostic performed by Engineer [2025-10-28 14:49:15]
Customer stated he is not currently at scanner and has limited details provided, 
Customer confirmed table will not drive up ...

  Case 2 | True label: cooling_system
  LLM resolution: recharged GC secondary loop to 2.1 bar and ordered the correct part for the leaking gradient coil secondary loop
  Text: REMOTE REMARKS:
*** Diagnostic performed by Engineer [2025-07-05 13:46:46]
Check logs and LCC GC side filling pressure shows 0,3 bar,
GC fill pressure dropped from 1,2 bar to 0,3 bar since 6/18/2025,
...

  Case 3 | True label: gradient_amp
  LLM resolution: Replaced the gradient coil and bus b

In [18]:
# ── Cell 6: Preview coil-opened and resolution for true positives ──

true_coil_detected = valid[(valid["llm_is_coil"] == True) & (valid["true_is_coil"] == True)]
print(f"True coil cases detected by LLM: {len(true_coil_detected)}")

print(f"\nCOIL OPENED DISTRIBUTION")
print("-" * 40)
print(true_coil_detected["llm_coil_opened"].value_counts().to_string())

print(f"\nRESOLUTION SAMPLES")
print("-" * 40)
for i, (_, row) in enumerate(true_coil_detected.head(10).iterrows()):
    opened = "OPENED" if row["llm_coil_opened"] == True else "NOT OPENED" if row["llm_coil_opened"] == False else "UNCLEAR"
    print(f"  [{opened}] {row['llm_resolution']}")

True coil cases detected by LLM: 99

COIL OPENED DISTRIBUTION
----------------------------------------
llm_coil_opened
False    76
True     15

RESOLUTION SAMPLES
----------------------------------------
  [NOT OPENED] Evaluated logs, ran dstream diagnostics, and performed coil networking and coil malfunction tests; root cause not determined
  [NOT OPENED] Replaced the posterior coil and tested the system; issue resolved.
  [NOT OPENED] restarted the system and verified the coils worked; issue resolved after reboot and test scan on shoulder coil
  [NOT OPENED] Issue cleared after cleaning the anterior coil connector/table socket and the system was able to scan normally; no further work needed.
  [NOT OPENED] Reviewed logs showing numerous posterior coil disconnect errors; power supply voltages were normal and a replacement posterior coil was ordered.
  [NOT OPENED] power cycled system; if issue persisted, replace defective DCP and/or posterior coil
  [NOT OPENED] Moved the anterior coi